# Fase 1 — Validação empírica da janela temporalA janela 2023–2025 foi escolhida por argumento: é posterior à migração dosistema de registro da SSP (R.D.O. → S.P.J., que a aba `METODOLOGIA` dosarquivos data "entre 2022 e 2023") e posterior ao choque de mobilidade dapandemia. Este notebook testa esse argumento contra os dados, em vez deapenas afirmá-lo. Entrega a **Figura 1**.**A regra de decisão, fixada antes de olhar o gráfico:** se houver um degrauvisível na virada de 2022 para 2023, a migração ainda estava em curso e ajanela encolhe para `[2024, 2025]` — o que custa uma linha em `config.py` euma reexecução. Se a série for contínua, 2023–2025 fica confirmado.**O controle interno que torna o teste informativo:** uma troca de sistema de*registro* mexe em como a ocorrência entra na base, não em quantos crimesacontecem. Ela deveria afetar mais as naturezas de alto volume e registroespontâneo (furto, roubo) do que o CVLI, que chega à estatística por outrocaminho e não deixa de ser contado porque o software mudou. Um degrau queapareça em *todas* as naturezas ao mesmo tempo é suspeito de ser artefato desistema; um que apareça só no volume é mais provavelmente registro.Roda sobre `data/processed/ssp_painel.csv`, que é **mensal**. Nenhummicrodado é reprocessado aqui.

In [ ]:
import sysfrom pathlib import Pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltRAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(RAIZ / "src"))from estilo import aplicar_estilo, salvar, AZUL, LARANJA, MUDO, TINTA_2, GRADEfrom config import SSP_PAINEL_CSV, SSP_COBERTURA_CSV, GRUPOS_NATUREZAfrom parse_ssp import normalizaaplicar_estilo()# Há mais de um Python instalado nesta máquina e nem todos têm as# dependências. Se der ImportError acima, o kernel do notebook está apontando# para o interpretador errado -- troque em "Select Kernel", no canto superior# direito, para o que este print mostrar quando funcionar.print(f"Python: {sys.executable}")painel = pd.read_csv(SSP_PAINEL_CSV, dtype={"codigo_ibge": str})cobertura = pd.read_csv(SSP_COBERTURA_CSV)print(painel.columns.tolist())print(f"{len(painel):,} linhas | anos {sorted(painel['ano'].unique())}")cobertura

## 1. Antes de qualquer gráfico: a série está completa?Um degrau falso aparece sozinho se algum mês estiver faltando. Duas checagens:todos os anos têm 12 meses, e quantas ocorrências têm mês desconhecido(`mes = 0`, gravado assim pelo `parse_ssp.py` para não perder a ocorrência dototal anual).

In [ ]:
sem_mes = painel.loc[painel["mes"] == 0, "ocorrencias"].sum()total = painel["ocorrencias"].sum()print(f"ocorrências com mês desconhecido: {sem_mes:,} de {total:,} "      f"({sem_mes / total:.3%})")contagem_meses = (painel[painel["mes"] > 0]                  .groupby("ano")["mes"].nunique().rename("meses_distintos"))print()print(contagem_meses.to_string())assert (contagem_meses == 12).all(), "algum ano não tem os 12 meses"

## 2. Figura 1 — série mensal estadual, 2022–2025**O eixo y começa em zero de propósito.** Num gráfico cujo objetivo éprocurar um degrau, cortar o eixo é o caminho mais curto para inventar um:qualquer oscilação vira penhasco quando se amplia a escala. Com a base emzero, o que aparece é a variação real em relação ao nível da série — e é essaa evidência honesta a favor ou contra a janela.

In [ ]:
mensal = (painel[painel["mes"] > 0]          .groupby(["ano", "mes"])["ocorrencias"].sum().reset_index())mensal["data"] = pd.to_datetime(    dict(year=mensal["ano"], month=mensal["mes"], day=1))mensal = mensal.sort_values("data")fig, ax = plt.subplots(figsize=(8.2, 3.4))ax.plot(mensal["data"], mensal["ocorrencias"], color=AZUL)# A fronteira sob teste: janeiro de 2023.ax.axvline(pd.Timestamp("2023-01-01"), color=LARANJA, lw=1.6, zorder=1)for ano in (2024, 2025):    ax.axvline(pd.Timestamp(f"{ano}-01-01"), color=GRADE, lw=1.0, zorder=0)ax.set_ylim(0)# Anotação na metade vazia do gráfico, longe da linha de dados.ax.annotate("jan/2023 — fronteira da janela\n(à direita, o recorte usado)",            xy=(pd.Timestamp("2023-02-01"), ax.get_ylim()[1] * 0.34),            ha="left", va="center", fontsize=8, color=LARANJA)ax.set_ylabel("ocorrências no mês")ax.set_title("Total mensal de ocorrências registradas no estado de São Paulo")fig.tight_layout()salvar(fig, "figura1_serie_mensal")plt.show()

## 3. A mesma série, por naturezaO total esconde composição: uma queda de furto pode compensar uma alta deroubo. Aqui estão as dez naturezas do bloco de criminalidade, cada uma na suaescala — o que interessa é a **forma** da série, não o nível.

In [ ]:
mapa = {normaliza(nat): grupo        for grupo, naturezas in GRUPOS_NATUREZA.items() for nat in naturezas}bloco = painel[(painel["mes"] > 0)].assign(    grupo=painel["natureza"].map(mapa)).dropna(subset=["grupo"])serie = (bloco.groupby(["grupo", "ano", "mes"])["ocorrencias"].sum()         .reset_index())serie["data"] = pd.to_datetime(    dict(year=serie["ano"], month=serie["mes"], day=1))grupos = list(GRUPOS_NATUREZA)fig, axes = plt.subplots(5, 2, figsize=(8.6, 9.2), sharex=True)for ax, g in zip(axes.ravel(), grupos):    s = serie[serie["grupo"] == g].sort_values("data")    ax.plot(s["data"], s["ocorrencias"], color=AZUL, lw=1.5)    ax.axvline(pd.Timestamp("2023-01-01"), color=LARANJA, lw=1.2)    ax.set_title(g, fontsize=9)    ax.set_ylim(0)    ax.tick_params(labelsize=7)fig.suptitle("Série mensal por natureza — a linha marca jan/2023", y=1.0)fig.tight_layout()salvar(fig, "figura1b_serie_por_natureza")plt.show()

## 4. O teste quantitativoO olho vê degraus onde há sazonalidade. A tabela abaixo compara a variaçãoano a ano de cada natureza. A pergunta não é "2022→2023 mudou?" — sempremuda — mas **"2022→2023 mudou fora do padrão das outras viradas de ano?"**.Se a migração de sistema tivesse contaminado 2023, esperaríamos ver2022→2023 destoando das viradas seguintes, e destoando na mesma direção emquase todas as naturezas.

In [ ]:
anual = (bloco.groupby(["grupo", "ano"])["ocorrencias"].sum()         .unstack("ano"))variacao = (anual.pct_change(axis=1) * 100).round(1)variacao.columns = [f"{a-1}→{a}" for a in variacao.columns]variacao = variacao.drop(columns=variacao.columns[0])variacao["|2022→2023|"] = variacao["2022→2023"].abs()variacao["mediana |demais|"] = (    variacao[["2023→2024", "2024→2025"]].abs().median(axis=1)).round(1)variacao.drop(columns="|2022→2023|")

In [ ]:
primeiro = variacao["2022→2023"]demais = variacao[["2023→2024", "2024→2025"]]print("Sinal da virada 2022→2023 por natureza:")print(f"  subiram: {(primeiro > 0).sum()} de {len(primeiro)}")print(f"  desceram: {(primeiro < 0).sum()} de {len(primeiro)}")print()print("Magnitude mediana das variações ano a ano:")print(f"  2022→2023: {primeiro.abs().median():.1f}%")print(f"  demais viradas: {demais.abs().median().median():.1f}%")print()fora = primeiro.abs() > 2 * demais.abs().median(axis=1)print("Naturezas em que 2022→2023 é mais que o dobro da variação típica:")print("  " + (", ".join(fora[fora].index) if fora.any() else "nenhuma"))

## 5. Decisão: janela 2023–2025 confirmadaAplicando a regra fixada no topo aos números acima:**A série não tem degrau em jan/2023.** O total mensal oscila em torno de 100mil ocorrências do início de 2022 ao fim de 2025, e a passagem de dezembro de2022 para janeiro de 2023 não se distingue das outras viradas de ano.**A virada 2022→2023 não tem direção comum.** Cinco naturezas sobem e cincodescem. Uma migração de sistema de registro que tivesse inflado ou deprimidoa captação teria empurrado quase todas para o mesmo lado.**A magnitude é a de sempre.** A variação mediana em 2022→2023 fica em tornode 6%, contra cerca de 6% nas viradas seguintes — dentro do ruído normal.**As duas exceções reforçam a conclusão em vez de enfraquecê-la.** As únicasnaturezas cuja variação em 2022→2023 passa do dobro da típica são `cvli` e`estupro_total` — precisamente as que *menos* dependem do sistema deregistro. Um artefato de software apareceria no volume de furto e roubo, queé registro espontâneo, e não no CVLI, que chega à estatística por outrocaminho. O padrão observado é o oposto do que a hipótese de artefatopreviria, o que aponta para variação criminal real.**Consequência prática:** `ANOS_JANELA = [2023, 2024, 2025]` fica como está,`config.py` não muda e a base não precisa ser regerada. 2022 permanece no`ssp_painel.csv` como série de controle — ele sustenta esta figura, mas estáfora da janela e não entra em `base_final.csv`.A Figura 1 entra nos Resultados Parciais documentando que o recorte foi**testado**, e não apenas escolhido — que é o que transforma uma decisão deconveniência em contribuição metodológica.